# Stats Notebook #2 — Testing Many Things at Once

You proved ~5% of noise tests pass p<0.05. Real studies test **hundreds** of
taxa → guaranteed false hits. Fix = **multiple-testing correction**.

Read → Run (`Shift+Enter`) → Modify → Explain.

## 1. Setup

In [1]:
import pandas as pd, numpy as np
from scipy import stats
df = pd.read_csv("example_gut_samples.csv")
taxa = ["Faecalibacterium","Bacteroides","Prevotella","Bifidobacterium","Escherichia","Other"]

## 2. Test every taxon: low_mood vs control

In [2]:
raw_p = []
for t in taxa:
    a = df[df.group=="low_mood"][t]; b = df[df.group=="control"][t]
    raw_p.append(stats.ttest_ind(a,b)[1])
for t,p in zip(taxa, raw_p):
    print(f"{t:16s} raw p = {p:.4f}  {'<0.05' if p<0.05 else ''}")

Faecalibacterium raw p = 0.0000  <0.05
Bacteroides      raw p = 0.5661  
Prevotella       raw p = 0.4066  
Bifidobacterium  raw p = 0.0054  <0.05
Escherichia      raw p = 0.5804  
Other            raw p = 0.1606  


## 3. The correction (Benjamini-Hochberg, by hand)

Idea: the more tests you run, the stricter the bar must be. BH scales each
p-value by `n_tests / rank`. numpy only — no extra installs.

In [3]:
def bh_correct(pvals):
    p = np.asarray(pvals); n = len(p)
    order = np.argsort(p); ranked = p[order]
    corr = ranked * n / np.arange(1, n+1)
    corr = np.minimum.accumulate(corr[::-1])[::-1]
    corr = np.clip(corr, 0, 1)
    out = np.empty(n); out[order] = corr
    return out

corr_p = bh_correct(raw_p)
res = pd.DataFrame({"taxon":taxa,"raw_p":np.round(raw_p,4),
                    "corrected_p":np.round(corr_p,4),
                    "still_significant":corr_p<0.05})
print(res.to_string(index=False))

           taxon  raw_p  corrected_p  still_significant
Faecalibacterium 0.0000       0.0000               True
     Bacteroides 0.5661       0.5804              False
      Prevotella 0.4066       0.5804              False
 Bifidobacterium 0.0054       0.0163               True
     Escherichia 0.5804       0.5804              False
           Other 0.1606       0.3212              False


### What you should see
`Faecalibacterium` and `Bifidobacterium` survive (corrected p<0.05) — real
signals. Everything else does not. Corrected p is always ≥ raw p: the price of
testing many things.

### EXPLAIN #1

*Why does the corrected p-value go UP, never down?*

> your answer here

## 4. 🔧 YOUR TURN

Change the comparison from `group` to `country` (random labels). Predict: after
correction, does ANY taxon survive?

In [4]:
raw_p2 = []
for t in taxa:
    a = df[df.country=="Italy"][t]; b = df[df.country=="Japan"][t]
    raw_p2.append(stats.ttest_ind(a,b)[1])
corr_p2 = bh_correct(raw_p2)
print(pd.DataFrame({"taxon":taxa,"raw_p":np.round(raw_p2,4),
                    "corrected_p":np.round(corr_p2,4),
                    "significant":corr_p2<0.05}).to_string(index=False))

           taxon  raw_p  corrected_p  significant
Faecalibacterium 0.0151       0.0909        False
     Bacteroides 0.1530       0.3059        False
      Prevotella 0.5551       0.6661        False
 Bifidobacterium 0.2579       0.3869        False
     Escherichia 0.7634       0.7634        False
           Other 0.0712       0.2137        False


### EXPLAIN #2

*Before correction the Italy/Japan Faecalibacterium gap looked significant. After correction?*

> your answer here

## Done

- Testing many things → false positives pile up.
- BH correction raises the bar; only strong signals survive.
- The fake country signal collapses; the real mood signals hold.

Save (Cmd+S). Next: real public data, or `curatedMetagenomicData`.